## tl;dr

The artifact recording contains **1,029 rows over 102.8 seconds at 10 Hz**, with no timestamp gaps or duplicate rows. Before inclusion, a model trained only on the shaky and mostly-still recordings mislabeled 1 of 98 artifact windows (1.02%). After recording-balanced hard-negative training, all 98 artifact windows are negative; the maximum shaky probability is 0.094. This is useful evidence for this one recording, not clinical validation.

## Context & Methods

This notebook checks the user-labeled artifact recording and documents its inclusion as a hard negative. The filename's `susp` token is ignored because the user explicitly labeled the run as artifacts.

### Key Assumptions

- All files are adult self-tests, not patient recordings.
- Grain is one paired two-sensor sample every 0.1 seconds.
- Five-second windows are used to distinguish sustained motion from isolated impulses.
- Features are recomputed from raw acceleration and gyro channels to remove calibration-version drift.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from train_movement_model import read_csv, make_windows, predict

SHAKY = Path(r'C:\Users\mwstr\Downloads\movement-sensors-1784658505915.csv')
STILL = Path(r'C:\Users\mwstr\Downloads\movement-sensors-1784658593112.csv')
ARTIFACT = Path(r'C:\Users\mwstr\Downloads\patient-3-susp-wk1-sensors.csv')
MODEL = Path('../local-models/movement-baseline.joblib')
REPORT = Path('../local-models/movement-baseline-report.json')

## Data

Profile row count, duration, cadence, gaps, and model windows for each recording.

In [ ]:
profiles = []
for label, path in [('shaky', SHAKY), ('mostly_still', STILL), ('artifact', ARTIFACT)]:
    timestamps, series, profile = read_csv(path)
    windows, _ = make_windows(timestamps, series)
    profiles.append({
        'label': label,
        'rows': profile['rows'],
        'duration_s': profile['duration_s'],
        'sample_interval_s': profile['median_sample_interval_s'],
        'gaps_over_0.15_s': profile['gaps_over_0_15_s'],
        'five_second_windows': len(windows),
    })
pd.DataFrame(profiles)

## Results

Compare the independent artifact holdout before inclusion with the final model after inclusion.

In [ ]:
report = json.loads(REPORT.read_text(encoding='utf-8'))
final_predictions = predict(MODEL, ARTIFACT)
probabilities = [row['shaky_probability'] for row in final_predictions]
summary = pd.DataFrame([
    {
        'stage': 'before artifact inclusion',
        'windows': report['artifact_holdout_before_inclusion']['windows'],
        'artifact_false_positive_rate': report['artifact_holdout_before_inclusion']['false_positive_rate_at_0_5'],
        'mean_shaky_probability': report['artifact_holdout_before_inclusion']['mean_shaky_probability'],
        'max_shaky_probability': report['artifact_holdout_before_inclusion']['max_shaky_probability'],
    },
    {
        'stage': 'after artifact inclusion',
        'windows': len(final_predictions),
        'artifact_false_positive_rate': sum(row['prediction'] == 'sustained_shaky' for row in final_predictions) / len(final_predictions),
        'mean_shaky_probability': sum(probabilities) / len(probabilities),
        'max_shaky_probability': max(probabilities),
    },
])
summary

## Takeaways

- The artifact CSV passes structural checks and is suitable as a hard-negative recording.
- The final model labels every five-second window in this recording as `not_shaky_or_artifact`.
- Recording-balanced weights prevent the 102.8-second artifact run from overpowering the two shorter recordings.
- The apparent perfect within-recording cross-validation is not independent validation because windows overlap and come from only three sessions. More separate shaky, still, and artifact sessions are required before estimating generalization.